# Google Images 의상 동물 이미지 수집 노트북

이 노트북은 `pet-images/images/cat` 와 `pet-images/images/dog` 폴더 안에 들어 있는 기존 반려동물 이미지 파일명을 읽어서 종 이름을 추출한 뒤, 각 종마다 상위 폴더 정보를 함께 사용해 Google Images에서 `"<species> cat clothes"` 또는 `"<species> dog clothes"`라는 검색어로 이미지를 찾고, 내려받은 결과를 `clothes-images/images/<species>` 폴더 구조로 저장하는 작업을 수행합니다.

전체 흐름은 다음과 같습니다.

1. 필요한 라이브러리를 불러옵니다.
2. 원본 경로, 저장 경로, 저장 개수 같은 실행 옵션을 설정합니다.
3. 파일명에서 종 이름을 추출합니다.
4. Google Images 검색과 이미지 저장에 필요한 함수를 정의합니다.
5. 한 종만 먼저 테스트해서 동작을 확인합니다.
6. 문제가 없으면 전체 종을 한 번에 실행합니다.
7. 마지막으로 저장 결과를 확인합니다.

즉, 이 노트북은 `설정 → 함수 정의 → 테스트 → 전체 실행 → 결과 확인` 순서로 따라가면서 크롤링 작업을 진행할 수 있도록 구성되어 있습니다.

## 1. 라이브러리 불러오기

이 셀은 노트북 전체에서 사용할 외부 라이브러리와 표준 라이브러리를 불러오는 단계입니다. 아래 라이브러리들은 각각 역할이 다릅니다.

- `re`: 파일명에서 종 이름을 뽑아낼 때 정규표현식을 사용하기 위해 필요합니다.
- `time`: Selenium 동작 사이에 잠시 대기해서 페이지가 로드될 시간을 주기 위해 사용합니다.
- `Path`: 경로를 문자열보다 안전하게 다루기 위해 사용합니다.
- `urlparse`: 이미지 URL에서 확장자 정보를 추론할 때 사용합니다.
- `requests`: Selenium이 찾아낸 실제 이미지 URL에 HTTP 요청을 보내 파일을 다운로드할 때 사용합니다.
- `selenium.webdriver`: Chrome 브라우저를 자동 제어하기 위해 사용합니다.
- `ElementClickInterceptedException`, `NoSuchElementException`, `TimeoutException`: 클릭 실패, 요소 미발견, 로딩 지연 같은 예외 상황을 처리하기 위해 사용합니다.
- `Options`, `Service`: Chrome 옵션 설정과 드라이버 실행 구성을 담당합니다.
- `By`, `Keys`: HTML 요소를 찾고 엔터 입력 같은 키보드 동작을 수행할 때 사용합니다.
- `EC`, `WebDriverWait`: 특정 요소가 나타날 때까지 기다리는 안정적인 브라우저 자동화를 위해 사용합니다.
- `ChromeDriverManager`: 로컬에 맞는 ChromeDriver를 자동으로 설치하거나 찾아서 브라우저 실행을 쉽게 해줍니다.

즉, 이 셀은 이후 셀들에서 사용할 모든 도구를 준비하는 역할을 합니다.

In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urlparse

import requests
from selenium import webdriver
from selenium.common.exceptions import ElementClickInterceptedException, NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

## 2. 경로와 실행 옵션 설정

이 셀은 크롤링 작업의 기본 설정값을 모아두는 구간입니다. 나중에 함수 내부를 수정하지 않고도, 이 셀의 값만 바꿔서 실행 조건을 조정할 수 있습니다.

각 변수의 의미는 다음과 같습니다.

- `SOURCE_DIR`: 기존 동물 이미지가 들어 있는 원본 루트 폴더입니다. 이 아래의 `cat`, `dog` 하위 폴더를 읽어서 종 목록과 동물 타입 정보를 함께 만듭니다.
- `OUTPUT_DIR`: 새로 다운로드한 의상 이미지를 저장할 루트 폴더입니다. 실제 저장은 `OUTPUT_DIR / species_name` 형태로 이루어집니다.
- `LIMIT_PER_SPECIES`: 종 하나당 몇 장까지 저장할지 정하는 값입니다. 예를 들어 `20`이면 종마다 최대 20장을 시도합니다.
- `MAX_SCROLLS`: Google Images 결과 화면을 아래로 몇 번 내릴지 정하는 값입니다. 값이 크면 더 많은 썸네일을 수집할 수 있지만 실행 시간도 늘어납니다.
- `PAUSE`: 검색, 클릭, 스크롤 사이에 몇 초 대기할지 정하는 값입니다. 사이트 로딩이 느리면 이 값을 늘리는 것이 안정적입니다.
- `HEADLESS`: `True`이면 브라우저 창을 화면에 띄우지 않고 백그라운드로 실행합니다. `False`이면 실제 Chrome 창이 보입니다.
- `OVERWRITE`: 이미 같은 이름의 파일이 있을 때 덮어쓸지 여부를 정합니다. `False`이면 기존 파일을 건너뜁니다.
- `IMAGE_EXTENSIONS`: 응답 헤더나 URL에서 확장자를 판별할 때 허용할 이미지 확장자 집합입니다.

마지막의 `OUTPUT_DIR.mkdir(...)`는 저장 폴더가 없더라도 자동으로 생성해주는 코드입니다. 즉, 이 셀은 전체 크롤링의 환경 설정과 출력 폴더 준비를 담당합니다.

In [3]:
SOURCE_DIR = Path.cwd().parent / "pet-images" / "images"
OUTPUT_DIR = Path("images")
LIMIT_PER_SPECIES = 20
MAX_SCROLLS = 12
PAUSE = 1.0
HEADLESS = False
OVERWRITE = False

IMAGE_EXTENSIONS = {".jpg", ".jpeg"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"SOURCE_DIR = {SOURCE_DIR.resolve()}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")

SOURCE_DIR = C:\Users\master\02_Workspace\Google_Pet To\pet-images\images
OUTPUT_DIR = C:\Users\master\02_Workspace\Google_Pet To\clothes-images\images


## 3. 종 목록 추출

이 셀의 목적은 원본 이미지 파일명에서 종 이름을 추출하고, 그 이미지가 `cat` 폴더에 있는지 `dog` 폴더에 있는지까지 함께 기록해서 이후 Google 검색에 사용할 목록을 만드는 것입니다.

여기서 정의된 함수는 다음 역할을 합니다.

- `discover_species_records(source_dir: Path) -> list[dict[str, str]]`
  이 함수는 `source_dir/cat`, `source_dir/dog` 아래에 있는 `.jpg` 파일들을 대상으로 삼고, 파일명에서 마지막 번호 부분을 제거해서 종 이름을 만든 뒤 그 종이 고양이인지 강아지인지도 함께 기록합니다. 예를 들어 `cat/Abyssinian_12.jpg`는 `species='Abyssinian'`, `animal='cat'`으로 정리됩니다.

함수 내부 동작은 다음과 같습니다.

- `(source_dir / animal).glob("*.jpg")`: `cat`, `dog` 폴더 각각에서 JPG 파일만 가져옵니다.
- `image_path.stem`: 확장자를 제외한 파일명만 가져옵니다.
- `image_path.stem.split("_")[:-1]`: 마지막 번호 부분을 제외한 앞쪽 조각들만 남깁니다.
- `"_".join(...)`: 앞쪽 조각들을 다시 이어 붙여 종 이름 문자열로 만듭니다.
- `record = {"species": ..., "animal": ...}`: 종 이름과 동물 타입을 같이 저장합니다.
- 집합처럼 중복을 제거해서 종별 기록 하나만 남깁니다.
- 마지막에는 `species`와 `animal` 기준으로 정렬된 목록을 반환합니다.

그 아래의 `species_records` 변수는 실제로 추출된 전체 종 목록을 담는 변수입니다. 각 원소는 `species`와 `animal` 키를 가지며, 이후 테스트 셀과 전체 실행 셀에서 기본 입력 목록으로 사용됩니다.

In [4]:
def discover_species_records(source_dir: Path) -> list[dict[str, str]]:
    seen = set()
    records = []

    for animal in ["cat", "dog"]:
        for image_path in (source_dir / animal).glob("*.jpg"):
            species_name = "_".join(image_path.stem.split("_")[:-1])
            if not species_name:
                continue

            key = (species_name, animal)
            if key in seen:
                continue

            seen.add(key)
            records.append({"species": species_name, "animal": animal})

    return sorted(records, key=lambda record: (record["animal"], record["species"]))


species_records = discover_species_records(SOURCE_DIR)
print(f"총 종 수: {len(species_records)}")
print(species_records)

총 종 수: 37
[{'species': 'Abyssinian', 'animal': 'cat'}, {'species': 'Bengal', 'animal': 'cat'}, {'species': 'Birman', 'animal': 'cat'}, {'species': 'Bombay', 'animal': 'cat'}, {'species': 'British_Shorthair', 'animal': 'cat'}, {'species': 'Egyptian_Mau', 'animal': 'cat'}, {'species': 'Maine_Coon', 'animal': 'cat'}, {'species': 'Persian', 'animal': 'cat'}, {'species': 'Ragdoll', 'animal': 'cat'}, {'species': 'Russian_Blue', 'animal': 'cat'}, {'species': 'Siamese', 'animal': 'cat'}, {'species': 'Sphynx', 'animal': 'cat'}, {'species': 'american_bulldog', 'animal': 'dog'}, {'species': 'american_pit_bull_terrier', 'animal': 'dog'}, {'species': 'basset_hound', 'animal': 'dog'}, {'species': 'beagle', 'animal': 'dog'}, {'species': 'boxer', 'animal': 'dog'}, {'species': 'chihuahua', 'animal': 'dog'}, {'species': 'english_cocker_spaniel', 'animal': 'dog'}, {'species': 'english_setter', 'animal': 'dog'}, {'species': 'german_shorthaired', 'animal': 'dog'}, {'species': 'great_pyrenees', 'animal': 'd

## 4. 크롤링에 필요한 함수 정의

이 셀은 실제 크롤링 작업의 핵심 기능들을 함수 단위로 정의하는 부분입니다. 각 함수는 하나의 역할에 집중하도록 나누어져 있어서, 테스트나 수정이 쉬운 구조입니다.

함수별 역할은 다음과 같습니다.

- `species_to_query(species_name: str, animal: str) -> str`
  종 이름과 동물 타입을 Google Images 검색어로 변환합니다. 예를 들어 `British_Shorthair`, `cat`이면 `British Shorthair cat clothes`가 되고, `beagle`, `dog`이면 `beagle dog clothes`가 됩니다.

- `build_driver(headless: bool = False)`
  Selenium이 제어할 Chrome 브라우저를 생성합니다. `headless`가 `True`면 창 없이 실행되고, `False`면 창이 보입니다. 내부에서 `Options`로 브라우저 옵션을 설정하고 `ChromeDriverManager`로 드라이버를 준비합니다.

- `accept_google_consent(driver, pause: float = 1.0) -> None`
  Google Images 접속 시 뜰 수 있는 쿠키 동의 창이나 약관 확인 버튼을 자동으로 눌러주는 함수입니다. 여러 가지 XPath를 순서대로 시도하며, 클릭 가능할 때만 처리합니다.

- `open_google_images(driver, pause: float = 1.0) -> None`
  Google Images 메인 페이지를 열고, 필요하면 동의 버튼 처리까지 이어서 수행합니다. 즉, 검색 전에 브라우저를 초기 상태로 맞추는 역할입니다.

- `search_google_images(driver, query: str, pause: float = 1.0) -> None`
  검색창을 찾아 `query`를 입력하고 엔터를 눌러 실제 검색을 수행합니다. 여기서 `query`는 `"Abyssinian clothes"` 같은 문자열입니다.

- `maybe_click_show_more(driver, pause: float = 1.0) -> None`
  스크롤을 내린 뒤 화면에 `Show more results` 버튼이 보이면 눌러서 더 많은 검색 결과를 불러옵니다. 결과 수가 적을 때 추가 데이터를 확보하는 역할을 합니다.

- `is_downloadable_image_url(url: str) -> bool`
  수집한 URL이 실제 다운로드 가능한 이미지 주소인지 판별합니다. `data:`로 시작하는 base64 썸네일이나 빈 문자열은 제외하고, `http://`, `https://` 주소만 통과시킵니다.

- `collect_image_urls(driver, limit: int, max_scrolls: int, pause: float = 1.0) -> list[str]`
  검색 결과 페이지에서 썸네일을 하나씩 클릭하면서 실제 원본 이미지 URL 후보를 수집합니다. `limit`은 몇 개까지 모을지, `max_scrolls`는 몇 번 스크롤할지 정합니다. 중복 URL은 `seen` 집합으로 제거합니다.

- `extension_from_response(response, url: str) -> str`
  다운로드 응답의 `content-type` 또는 URL 경로를 바탕으로 확장자를 추론합니다. 예를 들어 `image/png`면 `.png`, 알 수 없으면 기본적으로 `.jpg`를 사용합니다.

- `save_species_images(species_name: str, animal: str, image_urls: list[str], output_root: Path, overwrite: bool = False) -> int`
  수집한 `image_urls`에 대해 실제 HTTP 다운로드를 수행하고, `output_root/species_name` 폴더 안에 파일을 저장합니다. 파일명에는 동물 타입도 포함해서 저장하며, 저장 성공한 개수를 반환합니다.

이 셀에서 자주 쓰이는 매개변수들의 의미도 중요합니다.

- `driver`: Selenium이 제어하는 현재 Chrome 브라우저 객체입니다.
- `pause`: 각 동작 사이의 대기 시간입니다.
- `animal`: 현재 종이 고양이인지 강아지인지를 나타내는 문자열입니다.
- `query`: Google 검색창에 입력할 문자열입니다.
- `limit`: 수집할 최대 이미지 URL 수입니다.
- `max_scrolls`: 결과 페이지를 아래로 내리는 최대 횟수입니다.
- `response`: `requests.get()`으로 받은 HTTP 응답 객체입니다.
- `image_urls`: 실제 이미지 후보 URL 목록입니다.
- `output_root`: 저장 루트 폴더입니다.
- `overwrite`: 기존 파일 덮어쓰기 여부입니다.

즉, 이 셀은 이후 테스트와 전체 실행에서 재사용할 수 있는 기능 블록들을 준비하는 역할을 합니다.

In [ ]:
def species_to_query(species_name: str, animal: str) -> str:
    return f"{species_name.replace('_', ' ')} {animal} clothes"


def build_driver(headless: bool = False):
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")
    options.add_argument("--start-maximized")
    options.add_argument("--window-size=1600,1200")
    if headless:
        options.add_argument("--headless=new")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def accept_google_consent(driver, pause: float = 1.0) -> None:
    consent_xpaths = [
        "//button[.//div[text()='I agree']]",
        "//button[.//div[text()='Accept all']]",
        "//button[.='I agree']",
        "//button[.='Accept all']",
        "//button[contains(@aria-label, 'Accept')]",
        "//button[contains(@aria-label, 'agree')]",
        "//button[contains(@id, 'L2AGLb')]",
    ]
    for xpath in consent_xpaths:
        try:
            button = WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.XPATH, xpath)))
            button.click()
            time.sleep(pause)
            return
        except (TimeoutException, ElementClickInterceptedException):
            continue


def open_google_images(driver, pause: float = 1.0) -> None:
    driver.get("https://images.google.com/")
    time.sleep(pause)
    accept_google_consent(driver, pause)


def search_google_images(driver, query: str, pause: float = 1.0) -> None:
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.NAME, "q")))
    search_box.clear()
    search_box.send_keys(query)
    search_box.send_keys(Keys.ENTER)
    time.sleep(pause * 2)


def maybe_click_show_more(driver, pause: float = 1.0) -> None:
    show_more_xpaths = [
        "//input[@type='button' and @value='Show more results']",
        "//button[contains(., 'Show more results')]",
    ]
    for xpath in show_more_xpaths:
        try:
            button = driver.find_element(By.XPATH, xpath)
            driver.execute_script("arguments[0].click();", button)
            time.sleep(pause)
            return
        except NoSuchElementException:
            continue


def is_downloadable_image_url(url: str) -> bool:
    if not url or url.startswith("data:"):
        return False
    return url.startswith("http://") or url.startswith("https://")


def collect_image_urls(driver, limit: int, max_scrolls: int, pause: float = 1.0) -> list[str]:
    collected = []
    seen = set()

    for _ in range(max_scrolls):
        thumbnails = driver.find_elements(By.CSS_SELECTOR, "img.rg_i, img.YQ4gaf")
        for thumbnail in thumbnails:
            if len(collected) >= limit:
                return collected
            try:
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", thumbnail)
                time.sleep(pause / 2)
                driver.execute_script("arguments[0].click();", thumbnail)
                time.sleep(pause)
            except ElementClickInterceptedException:
                continue

            candidates = driver.find_elements(By.CSS_SELECTOR, "img.n3VNCb")
            for candidate in candidates:
                src = candidate.get_attribute("src") or ""
                if not is_downloadable_image_url(src) or src in seen:
                    continue
                seen.add(src)
                collected.append(src)
                if len(collected) >= limit:
                    return collected

        driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
        time.sleep(pause)
        maybe_click_show_more(driver, pause)

    return collected


def extension_from_response(response, url: str) -> str:
    content_type = response.headers.get("content-type", "").split(";")[0].strip().lower()
    content_map = {
        "image/jpeg": ".jpg",
        "image/png": ".png",
        "image/webp": ".webp",
        "image/gif": ".gif",
        "image/bmp": ".bmp",
    }
    if content_type in content_map:
        return content_map[content_type]

    suffix = Path(urlparse(url).path).suffix.lower()
    if suffix in IMAGE_EXTENSIONS:
        return suffix
    return ".jpg"


def save_species_images(species_name: str, animal: str, image_urls: list[str], output_root: Path, overwrite: bool = False) -> int:
    saved_count = 0
    species_dir = output_root / species_name
    species_dir.mkdir(parents=True, exist_ok=True)

    for index, image_url in enumerate(image_urls, start=1):
        try:
            response = requests.get(image_url, timeout=30, stream=True, headers={"User-Agent": "Mozilla/5.0"})
            response.raise_for_status()
            if not response.headers.get("content-type", "").lower().startswith("image/"):
                continue

            extension = extension_from_response(response, image_url)
            target_path = species_dir / f"{species_name}_{animal}_{index:03d}{extension}"
            if target_path.exists() and not overwrite:
                continue

            with target_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
            saved_count += 1
        except requests.RequestException:
            continue

    return saved_count

## 5. 한 종만 시험 실행하기

이 셀은 전체 종을 한 번에 실행하기 전에, 한 종만 먼저 테스트해보는 안전 점검 단계입니다. 브라우저가 정상적으로 열리는지, Google 검색이 되는지, URL 수집이 되는지, 이미지 저장이 되는지를 작은 범위에서 먼저 확인할 수 있습니다. 이제는 테스트 대상에 종 이름뿐 아니라 동물 타입 정보도 함께 포함됩니다.

각 변수의 의미는 다음과 같습니다.

- `TEST_RECORD`: 테스트용으로 선택한 한 개의 종 기록입니다. `species`와 `animal` 값을 함께 가집니다.
- `TEST_SPECIES`: 테스트 대상 종 이름입니다.
- `TEST_ANIMAL`: 테스트 대상 동물 타입입니다.
- `TEST_QUERY`: `TEST_SPECIES`와 `TEST_ANIMAL`을 실제 Google Images 검색어 형식으로 변환한 문자열입니다.
- `driver`: 테스트용 Chrome 브라우저 세션입니다.
- `test_urls`: 테스트 과정에서 수집된 이미지 URL 목록입니다.
- `saved_count`: 실제로 저장에 성공한 이미지 개수입니다.

이 셀의 핵심 목적은 전체 작업 전에 실패 지점을 빨리 찾는 것입니다. 예를 들어 `saved_count`가 0이면 Google DOM 구조, 네트워크 상태, 동의 창 처리, 이미지 URL 판별 로직 등을 먼저 점검해야 한다는 신호가 됩니다.

In [ ]:
TEST_RECORD = species_records[0]
TEST_SPECIES = TEST_RECORD["species"]
TEST_ANIMAL = TEST_RECORD["animal"]
TEST_QUERY = species_to_query(TEST_SPECIES, TEST_ANIMAL)

driver = build_driver(headless=HEADLESS)
open_google_images(driver, PAUSE)
search_google_images(driver, TEST_QUERY, PAUSE)
test_urls = collect_image_urls(driver, limit=5, max_scrolls=3, pause=PAUSE)
saved_count = save_species_images(TEST_SPECIES, TEST_ANIMAL, test_urls, OUTPUT_DIR, overwrite=OVERWRITE)

print(f"test species = {TEST_SPECIES}")
print(f"test animal = {TEST_ANIMAL}")
print(f"candidate urls = {len(test_urls)}")
print(f"saved images = {saved_count}")

## 6. 테스트용 브라우저 닫기

앞의 테스트 셀에서 열었던 Selenium 브라우저를 종료하는 정리 단계입니다.

- `driver.quit()`는 현재 브라우저 창만 닫는 것이 아니라, Selenium 세션 전체를 종료합니다.
- 이 작업을 해두면 메모리 점유와 Chrome 프로세스가 불필요하게 남는 것을 막을 수 있습니다.

즉, 이 셀은 다음 전체 실행 전에 테스트 세션을 깨끗하게 정리하는 역할을 합니다.

In [ ]:
driver.quit()

## 7. 전체 종 일괄 수집 실행

이 셀은 실제 본작업을 수행하는 셀입니다. `TARGET_RECORDS`에 들어 있는 모든 종을 순회하면서 검색, URL 수집, 다운로드를 반복합니다.

주요 변수의 의미는 다음과 같습니다.

- `TARGET_RECORDS`: 이번 실행에서 처리할 종 기록 목록입니다. 기본값은 전체 종 목록인 `species_records`입니다.
- `record`: 반복문에서 현재 처리 중인 종 기록입니다.
- `species_name`: 현재 처리 중인 종 이름입니다.
- `animal`: 현재 처리 중인 동물 타입입니다.
- `query`: 현재 종과 동물 타입에 대해 생성된 Google Images 검색어입니다.
- `image_urls`: 현재 종에 대해 수집된 다운로드 후보 URL 목록입니다.
- `saved_count`: 해당 종에서 실제 저장된 이미지 수입니다.
- `driver`: 전체 실행 동안 재사용하는 브라우저 세션입니다.

실행 순서는 다음과 같습니다.

1. 종 이름과 동물 타입을 검색어로 바꿉니다.
2. Google Images를 엽니다.
3. 검색어를 입력해 결과 페이지로 이동합니다.
4. 썸네일을 클릭하며 이미지 URL을 모읍니다.
5. 수집한 URL들을 종별 폴더에 저장합니다.
6. 현재 종의 저장 결과를 출력합니다.

또한 `try ... finally` 구조를 사용했기 때문에, 중간에 오류가 나더라도 마지막에 `driver.quit()`가 실행되어 브라우저가 정리되도록 설계되어 있습니다.

In [ ]:
TARGET_RECORDS = species_records
# 예시: TARGET_RECORDS = [record for record in species_records if record["animal"] == "cat"]

driver = build_driver(headless=HEADLESS)

try:
    for record in TARGET_RECORDS:
        species_name = record["species"]
        animal = record["animal"]
        query = species_to_query(species_name, animal)
        print(f"\n[{species_name}] Searching: {query}")

        open_google_images(driver, PAUSE)
        search_google_images(driver, query, PAUSE)
        image_urls = collect_image_urls(driver, LIMIT_PER_SPECIES, MAX_SCROLLS, PAUSE)
        saved_count = save_species_images(species_name, animal, image_urls, OUTPUT_DIR, overwrite=OVERWRITE)

        print(f"[{species_name}] Collected {len(image_urls)} candidate URLs, saved {saved_count} images.")
finally:
    driver.quit()

## 8. 저장 결과 확인

이 셀은 다운로드가 끝난 뒤 결과를 요약해서 확인하는 단계입니다. 어떤 종 폴더가 생성되었는지, 각 폴더 안에 몇 개의 파일이 들어 있는지 빠르게 볼 수 있습니다.

주요 변수의 의미는 다음과 같습니다.

- `saved_summary`: `{종이름: 저장된 파일 수}` 형태의 딕셔너리입니다.
- `species_dir`: `OUTPUT_DIR` 아래에 있는 각 종 폴더를 가리키는 경로 객체입니다.

이 셀은 크롤링이 제대로 되었는지 1차 확인하는 용도로 유용합니다. 예를 들어 특정 종의 파일 수가 0이거나 유난히 적으면, 그 종 검색 결과가 부족했는지 또는 중간 다운로드 실패가 많았는지 점검해볼 수 있습니다.

In [ ]:
saved_summary = {}
for species_dir in sorted(OUTPUT_DIR.iterdir()):
    if species_dir.is_dir():
        saved_summary[species_dir.name] = len(list(species_dir.glob("*")))

saved_summary